In [7]:
import geopandas as gpd
import joblib
import numpy as np
import odc.geo  # noqa: F401
import pandas as pd
from shapely import geometry
from sklearn.ensemble import RandomForestClassifier

from utils import load_data

In [8]:
%reload_ext autoreload
%autoreload 2

## Find and load S2 data

Load data and set up your array to use for prediction

In [18]:
# Configure some things up front
chunks = dict(x=256, y=256)
datetime = "2022"

bbox = [177.14, -18.41, 178.80, -17.2]
bbox_geometry = geometry.box(*bbox)

gdf = gpd.GeoDataFrame({'geometry': [bbox_geometry]}, crs='EPSG:4326')
gdf.explore()

In [22]:
merged = load_data(bbox, chunks=dict(x=256, y=256), resolution=10)
merged

Successfully found and loading 6 tiles for the year 2022...
Searching for Sentinel-1 radar data...
No Sentinel-1 radar composites for 2022 in this bounding box.


<xarray.Dataset> Size: 31GB
Dimensions:        (y: 14062, x: 18480)
Coordinates:
  * y              (y) float64 112kB -1.931e+06 -1.932e+06 ... -2.072e+06
  * x              (x) float64 148kB 3.021e+06 3.021e+06 ... 3.206e+06 3.206e+06
    spatial_ref    int32 4B 3832
    time           datetime64[ns] 8B 2022-01-01
Data variables: (12/25)
    blue           (y, x) uint16 520MB dask.array<chunksize=(256, 256), meta=np.ndarray>
    green          (y, x) uint16 520MB dask.array<chunksize=(256, 256), meta=np.ndarray>
    red            (y, x) uint16 520MB dask.array<chunksize=(256, 256), meta=np.ndarray>
    rededge1       (y, x) uint16 520MB dask.array<chunksize=(256, 256), meta=np.ndarray>
    rededge2       (y, x) uint16 520MB dask.array<chunksize=(256, 256), meta=np.ndarray>
    rededge3       (y, x) uint16 520MB dask.array<chunksize=(256, 256), meta=np.ndarray>
    ...             ...
    ndwi           (y, x) float64 2GB dask.array<chunksize=(256, 256), meta=np.ndarray>
    ndre           (y, x) float64 2GB dask.array<chunksize=(256, 256), meta=np.ndarray>
    elevation      (y, x) float32 1GB dask.array<chunksize=(256, 256), meta=np.ndarray>
    elevation_std  (y, x) float32 1GB dask.array<chunksize=(257, 257), meta=np.ndarray>
    slope          (y, x) float32 1GB nan nan 0.0 0.0 0.0 ... nan nan nan nan
    aspect         (y, x) float32 1GB nan nan 90.0 90.0 90.0 ... nan nan nan nan

## Train and predict

When you change your training data, you can re-train and predict here.

In [23]:
training_file = "training_data/mred-tdata.geojson"

tdata = gpd.read_file(training_file, bbox=bbox_geometry)
tdata.explore()

In [6]:
# Get values for each of the image bands at each of the points.
pts_proj = tdata.to_crs(merged.odc.crs)

# a DataArray with x & y coords
pts_da = pts_proj.assign(x=pts_proj.geometry.x, y=pts_proj.geometry.y).to_xarray()

# a dataframe or series (for a single point)
pt_values_i = (
    merged.sel(pts_da[["x", "y"]], method="nearest").squeeze().compute().to_pandas()
)

if isinstance(pt_values_i, pd.Series):
    pt_values_i = pt_values_i.to_frame().transpose()
    pt_values_i.index = tdata.index

NameError: name 'merged' is not defined

In [ ]:
training_array = pd.concat([tdata, pt_values_i], axis=1).to_crs(4326)
training_array = training_array.drop(
    columns=[
        "y",
        "x",
        "spatial_ref",
        "time",
        "fid",
        "index",
        "lulc_class",
        "path",
        "geometry",
        "layer",
        "id",
    ]
)
# Drop rows where there are any NaNs
training_array = training_array.dropna()

training_array.head()

In [ ]:
classifier = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=10,
    n_jobs=-1,
    random_state=42,
)

training_data = np.array(training_array)[:, 1:]
classes = np.array(training_array)[:, 0]

model = classifier.fit(training_data, classes)


In [ ]:
joblib.dump(model, "test_model.dump")

In [ ]:
# Print feature importances against column headings
fields_importances = sorted(
    zip(training_array.columns[1:], classifier.feature_importances_),
    key=lambda x: x[1],
    reverse=True,
)

for i in fields_importances:
    # Format as a table to 2 decinal places
    print(f"{i[0]:<11}| {i[1]:.2f}")
